# Bathymetry — coastal depth profile

Download a small GEBCO 2020 subset across the Iberian continental margin and
draw a west-east depth transect — the classic shelf → slope → abyssal-plain
cross-section a user would pull to size a cable route or a survey line.

In [ ]:
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from pyramids.dataset import Dataset

from earthlens.core import EarthLens

## Download the margin

An AOI off Lisbon spanning the shelf (east) out to deep water (west).

In [ ]:
out = tempfile.mkdtemp()
paths = EarthLens(
    data_source='gebco',
    dataset='gebco_2020',
    aoi=[-11.0, 38.4, -9.3, 38.8],
    path=out,
).download(progress_bar=False)

ds = Dataset.read_file(paths[0])
arr = np.asarray(ds.read_array(), dtype='float32')
arr = arr[0] if arr.ndim == 3 else arr
arr = np.where(arr == 32767, np.nan, arr)
print('grid', arr.shape, '| epsg', ds.epsg)

## Take the central transect

The middle row is a west-east line of elevations; map the column index onto
longitude using the AOI bounds.

In [ ]:
transect = arr[arr.shape[0] // 2, :]
lons = np.linspace(-11.0, -9.3, transect.size)
shelf_break = lons[np.nanargmin(np.abs(transect + 200))]  # ~200 m isobath
print('depth range', round(np.nanmin(transect)), '..', round(np.nanmax(transect)), 'm')

In [ ]:
plt.figure(figsize=(7, 4))
plt.fill_between(lons, transect, np.nanmin(transect), color='#9ecae1')
plt.plot(lons, transect, color='#08519c')
plt.axhline(0, color='k', lw=0.8)
plt.axvline(shelf_break, color='crimson', ls='--', lw=1, label='~200 m shelf break')
plt.xlabel('longitude (deg)')
plt.ylabel('elevation (m)')
plt.title('GEBCO 2020 west-east depth profile — Iberian margin')
plt.legend()
plt.show()